# Flow State Persistence [Step 2 - Checkpointing and Resuming Flows]

> **MLCourse - Agentic AI - CrewAI Flows and Orchestration**

CrewAI Flows support state persistence via the `@persist` decorator.
When applied to a Flow class, every step's state is checkpointed to a
SQLite database. This lets you resume a flow from the last checkpoint,
fork a flow into a new run, and inspect state between runs.

## What you will learn

1. The `@persist` decorator and how it checkpoints flow state.
2. Using `CheckpointConfig` to specify database location.
3. Resuming a flow from a checkpoint with `restore_from_state_id`.
4. Forking a flow into a new run that preserves the source history.
5. Inspecting persisted state between runs.

## Key takeaways

- `@persist` writes state after every step to SQLite automatically.
- You can resume a crashed or interrupted flow without re-running completed steps.
- Forking creates a new run ID while hydrating from a previous run's state.
- Persistence is optional -- flows work fine without it.

In [ ]:
# --- Standard library imports -------------------------------------------------
import os                           # Environment variable access
import time                         # Delays for demo purposes
from pathlib import Path            # OOP path handling

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv      # Load .env into os.environ

# Walk up to track root
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Setup complete. Track root resolved to:", TRACK)

## 1 -- Verify Ollama availability

In [ ]:
from langchain_ollama import ChatOllama

try:
    _test = ChatOllama(model="llama3.1:8b", temperature=0)
    _test.invoke("ping")
    LLM_AVAILABLE = True
    print("[GREEN] Ollama reachable -- full pipeline will run")
except Exception as exc:
    LLM_AVAILABLE = False
    print("[WARN] Ollama not reachable:", exc)
    print("Flow structure demonstrated without LLM calls")

## 2 -- Import CrewAI persistence classes

In [ ]:
from pydantic import BaseModel
from crewai import Flow, Agent, Task, Crew
from crewai.flow import start, listen, persist
from crewai import CheckpointConfig

print("CrewAI persistence imports successful")

## 3 -- Define typed state with a checkpoint ID

When using `@persist`, the flow state gets a unique `id` field automatically
(inherited from `FlowState`). You can add additional fields to track
intermediate results. The checkpoint database stores state snapshots keyed
by this ID.

In [ ]:
class PipelineState(BaseModel):
    """State for a multi-step pipeline with persistence."""
    topic: str = ""          # Input topic
    research: str = ""       # Research output
    summary: str = ""        # Summary output
    iteration: int = 0       # Track how many times this flow ran

print("State model defined:", list(PipelineState.model_fields.keys()))

## 4 -- Define agents and tasks

In [ ]:
llm = ChatOllama(model="llama3.1:8b", temperature=0) if LLM_AVAILABLE else None

analyst = Agent(
    role="Data Analyst",
    goal="Analyze the given topic and produce key findings",
    backstory="You are a data analyst who extracts actionable insights.",
    llm=llm,
    verbose=False,
)

summarizer = Agent(
    role="Summary Writer",
    goal="Condense findings into a concise executive summary",
    backstory="You are an expert at distilling complex information.",
    llm=llm,
    verbose=False,
)

analysis_task = Task(
    description="Analyze this topic and list key findings: {topic}",
    expected_output="A bulleted list of 5-7 key findings.",
    agent=analyst,
)

summary_task = Task(
    description="Summarize these findings into a 2-3 sentence executive summary: {research}",
    expected_output="A concise 2-3 sentence summary.",
    agent=summarizer,
)

print("Agents and tasks defined")

## 5 -- Build a persisted flow

The `@persist` decorator is applied to the Flow class. This tells CrewAI
to checkpoint the state to SQLite after every step completes. You can
optionally pass a `CheckpointConfig` to control the database path.

In [ ]:
# Define a checkpoint database path in the temp directory
CHECKPOINT_DB = Path(os.environ.get("TEMP", "/tmp")) / "crewai_checkpoints.db"
print(f"Checkpoint database will be at: {CHECKPOINT_DB}")


@persist(verbose=True)
class PersistedPipeline(Flow[PipelineState]):
    """A 2-step pipeline with automatic state checkpointing.

    After each step completes, the full state is written to SQLite.
    If the flow is interrupted, it can resume from the last checkpoint.
    """

    @start()
    def analyze(self):
        """Step 1: Analyze the topic."""
        self.state.iteration += 1
        print(f"[Persist] Analysis step (iteration {self.state.iteration})")
        crew = Crew(
            agents=[analyst],
            tasks=[analysis_task],
            verbose=False,
        )
        result = crew.kickoff(inputs={"topic": self.state.topic})
        self.state.research = str(result)
        print(f"[Persist] Analysis complete ({len(self.state.research)} chars)")
        return self.state.research

    @listen(analyze)
    def summarize(self, research_result):
        """Step 2: Summarize the research."""
        print(f"[Persist] Summarize step")
        crew = Crew(
            agents=[summarizer],
            tasks=[summary_task],
            verbose=False,
        )
        result = crew.kickoff(inputs={"research": research_result})
        self.state.summary = str(result)
        print(f"[Persist] Summary complete ({len(self.state.summary)} chars)")
        return self.state.summary

print("Persisted flow class defined")

## 6 -- Run the flow (first execution)

When `@persist` is active, each step writes its state to the checkpoint
database. The flow's `state.id` is a UUID that identifies this run.

In [ ]:
if LLM_AVAILABLE:
    flow1 = PersistedPipeline()
    result1 = flow1.kickoff(inputs={"topic": "Machine learning model deployment strategies"})

    # Capture the flow ID for later resumption
    flow1_id = flow1.state.id
    print(f"\nFlow 1 complete. State ID: {flow1_id}")
    print(f"Topic:    {flow1.state.topic}")
    print(f"Iteration: {flow1.state.iteration}")
    print(f"Research:  {flow1.state.research[:150]}...")
    print(f"Summary:   {flow1.state.summary[:150]}...")
else:
    print("Ollama not available -- demonstrating persistence API only")
    flow1 = PersistedPipeline()
    print("Flow class ready for execution when Ollama is available")

## 7 -- Resume from checkpoint

You can resume a flow from its last persisted state using the
`restore_from_state_id` parameter. This hydrates `self.state` from
the database and starts execution at the first step. The new run gets
a fresh `state.id` but inherits all other state values.

In [ ]:
if LLM_AVAILABLE and flow1_id:
    print(f"Resuming flow from state ID: {flow1_id}")
    flow2 = PersistedPipeline()
    result2 = flow2.kickoff(
        inputs={"topic": "Updated topic: Edge deployment strategies"},
        restore_from_state_id=flow1_id,
    )
    print(f"\nFlow 2 (resumed) complete. New state ID: {flow2.state.id}")
    print(f"Iteration: {flow2.state.iteration}")
    print(f"Note: The state was hydrated from the previous run's checkpoint")
else:
    print("Skipping resume demo -- Ollama not available or no previous run")

## 8 -- Fork into a new run

Forking creates a new run that starts from a previous run's state but
gets its own unique ID. The source flow's history is preserved. This is
useful for "what-if" scenarios where you want to branch from a checkpoint.

In [ ]:
if LLM_AVAILABLE and flow1_id:
    print(f"Forking from state ID: {flow1_id}")
    flow3 = PersistedPipeline()
    result3 = flow3.kickoff(
        inputs={"topic": "Forked topic: Serverless ML inference"},
        restore_from_state_id=flow1_id,
    )
    print(f"\nFlow 3 (forked) complete. Forked state ID: {flow3.state.id}")
    print(f"Original ID: {flow1_id}")
    print(f"New ID:      {flow3.state.id} (different -- fork preserves source)")
else:
    print("Skipping fork demo -- Ollama not available or no previous run")

## 9 -- Manual checkpoint inspection

The checkpoint database is SQLite. You can inspect it directly to see
what was persisted. Each row contains the flow state serialized as JSON,
along with metadata like timestamps and state IDs.

In [ ]:
import sqlite3

if CHECKPOINT_DB.exists():
    conn = sqlite3.connect(str(CHECKPOINT_DB))
    cursor = conn.cursor()

    # List all tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = cursor.fetchall()
    print("Tables in checkpoint DB:", [t[0] for t in tables])

    # Inspect the flow states
    for table_name in [t[0] for t in tables]:
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cursor.fetchone()[0]
        print(f"  {table_name}: {count} rows")

        if count > 0 and table_name != "sqlite_sequence":
            cursor.execute(f"SELECT * FROM {table_name} LIMIT 1")
            row = cursor.fetchone()
            cols = [desc[0] for desc in cursor.description]
            print(f"  Columns: {cols}")
            for col, val in zip(cols, row):
                val_str = str(val)[:100] if val else "NULL"
                print(f"    {col}: {val_str}")

    conn.close()
else:
    print(f"Checkpoint DB not found at {CHECKPOINT_DB}")
    print("Run the flow first to create checkpoints")

## 10 -- Summary

The `@persist` decorator gives you:

- **Automatic checkpointing**: State is written to SQLite after every step.
- **Resumability**: Resume from `restore_from_state_id` to skip completed steps.
- **Forking**: Create new runs from any previous state without altering the source.
- **Inspection**: Direct SQLite access for debugging and auditing.

This is invaluable for long-running flows that might crash, or for
experimentation where you want to branch from a known state.

In [ ]:
print("=" * 60)
print("MODULE SUMMARY -- Flow State Persistence")
print("=" * 60)
print()
print("Decorator:")
print("  @persist(verbose=True)  -- checkpoint state after every step")
print()
print("Parameters:")
print("  restore_from_state_id  -- resume from a previous run's state")
print()
print("Key classes:")
print("  CheckpointConfig        -- configure checkpoint database path")
print("  Flow[StateType]         -- base class with typed state")
print()
print("Execution pattern:")
print("  flow.kickoff(inputs={...})                     -- first run")
print("  flow.kickoff(restore_from_state_id='uuid')     -- resume")
print("  flow.kickoff(restore_from_state_id='uuid')     -- fork (new ID)")
print()
print("Checkpoint storage:")
print("  SQLite database (default location or custom via CheckpointConfig)")